In [175]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


#### Getting the data

In [176]:
def load_with_source(folder_path, source_name):
    folder = Path(folder_path)
    dfs = []
    for file in folder.glob("*.csv"):
        df = pd.read_csv(file)
        df["source"] = source_name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [177]:
# getting the data
acm = load_with_source("data/raw/ACM", "ACM")
ieee = load_with_source("data/raw/IEEE", "IEEE")
springer = load_with_source("data/raw/Springer", "Springer")

sd = pd.read_csv("data/raw/ScienceDirect/ScienceDirect_query01.csv")
sd["source"] = "ScienceDirect"

In [178]:
# create table to keep track of counts
count_columns = ["data", "ACM", "IEEE", "Springer", "ScienceDirect"]

count_db = pd.DataFrame(columns = count_columns)
count_db.loc[0] = ["raw", len(acm), len(ieee), len(springer), len(sd)]
count_db

,data,ACM,IEEE,Springer,ScienceDirect
0,raw,78,337,1499,49


In [179]:
# getting column names for all DB
acm_col = list(acm)
ieee_col = list(ieee)
springer_col = list(springer)
sd_col = list(sd)

In [180]:
print("acm", acm_col)
print("ieee", ieee_col)
print("springer", springer_col)
print("sd", sd_col)

acm ['Key', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract Note', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting Name', 'Conference Name', 'Court', '

### prepare the individual DB so they can be merged into one later

##### we are with making sure all db only have for journal articles and conference papers / proceedings

In [181]:
# create function for filter for journal articles and conferences papers 
def filter_journal_conference(df, column_name):
    '''
    This funciton...
    '''
    # define regex pattern
    pattern = re.compile(r"conference|journal|article", re.IGNORECASE)

    # ensure all entries are strings and strip whitespace
    cleaned_col = df[column_name].astype(str).str.strip()

    # filter for rows matching the pattern
    filtered_df = df[cleaned_col.str.contains(pattern, na = False)]

    return filtered_df

In [182]:
acm_type_filtered = filter_journal_conference(acm, "Item Type")
ieee_type_filtered = filter_journal_conference(ieee, "Document Identifier")
springer_type_filtered = filter_journal_conference(springer, "Content Type")
sd_type_filtered = filter_journal_conference(sd, "Item Type")

In [183]:
# update counting DB
type_filter_row = pd.DataFrame([["type filtered", len(acm_type_filtered), len(ieee_type_filtered),
                            len(springer_type_filtered), len(sd_type_filtered)]],
                            columns = count_db.columns)

count_db = pd.concat([count_db, type_filter_row], ignore_index = True)

count_db

,data,ACM,IEEE,Springer,ScienceDirect
0,raw,78,337,1499,49
1,type filtered,76,337,1499,49


##### next, filter for entires containing "fall detection" AND ("privacy" OR "security") in title OR abstract